In [1]:
import pandas as pd
df = pd.read_csv("../data/wiki_movie_plots_deduped.csv")
print(df.shape)
print(df.columns.tolist())
print(df.isna().sum())
print(df.iloc[:, 2].value_counts().head(15))

(34886, 8)
['Release Year', 'Title', 'Origin/Ethnicity', 'Director', 'Cast', 'Genre', 'Wiki Page', 'Plot']
Release Year           0
Title                  0
Origin/Ethnicity       0
Director               0
Cast                1422
Genre                  0
Wiki Page              0
Plot                   0
dtype: int64
Origin/Ethnicity
American        17377
British          3670
Bollywood        2931
Tamil            2599
Telugu           1311
Japanese         1188
Malayalam        1095
Hong Kong         791
Canadian          723
Australian        576
South_Korean      522
Chinese           463
Kannada           444
Bengali           306
Russian           232
Name: count, dtype: int64


In [2]:
df["plot_words"] = df["Plot"].str.split().str.len()
print(df["plot_words"].describe())
print("short plots:", (df["plot_words"] < 50).sum())
print("duplicate title+year:", df.duplicated(subset=["Title", "Release Year"]).sum())

keep = ["American", "Bollywood", "Tamil", "Telugu", "Malayalam", "Kannada"]
sub = df[df["Origin/Ethnicity"].isin(keep)].copy()
print(sub.groupby("Origin/Ethnicity")["plot_words"].median())
sub.sample(5, random_state=0)[["Title", "Plot"]]

count    34886.000000
mean       372.493206
std        315.753223
min          2.000000
25%        122.000000
50%        284.000000
75%        581.000000
max       6752.000000
Name: plot_words, dtype: float64
short plots: 2803
duplicate title+year: 278
Origin/Ethnicity
American     383.0
Bollywood    243.0
Kannada      163.5
Malayalam    180.0
Tamil        208.0
Telugu       252.0
Name: plot_words, dtype: float64


,Title,Plot
13482,The Hurricane,The film tells the story of middleweight boxer...
5407,Iron Man,Chandler plays a coal miner who is encouraged ...
5617,Everything I Have Is Yours,"The husband and wife dance team, Chuck and Pam..."
13623,Cast Away,"In December 1995, Chuck Noland is a time-obses..."
16926,Black or White,Elliot Anderson (Kevin Costner) is widowed aft...


In [3]:
dups = sub[sub.duplicated(subset=["Title", "Release Year"], keep=False)]
print(len(dups))
sub = sub.drop_duplicates(subset=["Title", "Release Year"], keep="first").reset_index(drop=True)
sub.to_csv("../data/clean.csv", index=False)
print(sub.shape)

184
(25663, 9)


In [4]:
sub = sub[sub["plot_words"] >= 50].reset_index(drop=True)
sub.to_csv("../data/clean.csv", index=False)
print(sub.shape)

(24076, 9)


In [5]:
dups = sub[sub.duplicated(subset=["Title", "Release Year"], keep=False)]
print("rows involved in duplicates:", len(dups))

sub = sub.drop_duplicates(subset=["Title", "Release Year"], keep="first").reset_index(drop=True)

sub.to_csv("../data/clean.csv", index=False)
print("final shape:", sub.shape)

rows involved in duplicates: 0
final shape: (24076, 9)


In [6]:
candidates = ["Kaakha Kaakha", "Gharshana", "Sethu", "Tere Naam", "Pokkiri",
              "Wanted", "Lage Raho", "Shankar Dada Zindabad", "Kaththi", "Khaidhi"]
for name in candidates:
    print("-----", name)
    hits = sub[sub["Title"].str.contains(name, case=False)]
    print(hits[["Title", "Release Year", "Origin/Ethnicity"]].to_string())

----- Kaakha Kaakha
               Title  Release Year Origin/Ethnicity
21942  Kaakha Kaakha          2003            Tamil
----- Gharshana
              Title  Release Year Origin/Ethnicity
23496     Gharshana          2004           Telugu
23797  Sangharshana          2011           Telugu
----- Sethu
                Title  Release Year Origin/Ethnicity
21460  Sethupathi IPS          1994            Tamil
21759           Sethu          1999            Tamil
22733      Sethupathi          2016            Tamil
----- Tere Naam
                   Title  Release Year Origin/Ethnicity
18216          Tere Naam          2003        Bollywood
18778  Zindagi Tere Naam          2012        Bollywood
----- Pokkiri
                              Title  Release Year Origin/Ethnicity
19954  Pokkiri Raja ( പോക്കിരിരാജ )          2010        Malayalam
20419                 Pokkiri Simon          2017        Malayalam
21107                  Pokkiri Raja          1982            Tamil
22137            

In [7]:
pairs = pd.DataFrame(
    [
        ("Kaakha Kaakha", 2003, "Gharshana", 2004),
        ("Sethu", 1999, "Tere Naam", 2003),
        ("Pokkiri", 2006, "Wanted", 2009),
        ("Lage Raho Munna Bhai", 2006, "Shankar Dada Zindabad", 2007),
        ("Kaththi", 2014, "Khaidhi No. 150", 2017),
    ],
    columns=["original_title", "original_year", "remake_title", "remake_year"],
)
pairs.to_csv("../data/remake_pairs.csv", index=False)
print(pairs)

         original_title  original_year           remake_title  remake_year
0         Kaakha Kaakha           2003              Gharshana         2004
1                 Sethu           1999              Tere Naam         2003
2               Pokkiri           2006                 Wanted         2009
3  Lage Raho Munna Bhai           2006  Shankar Dada Zindabad         2007
4               Kaththi           2014        Khaidhi No. 150         2017


In [8]:
pairs = pd.read_csv("../data/remake_pairs.csv")
for _, r in pairs.iterrows():
    for kind in ["original", "remake"]:
        hit = sub[(sub["Title"] == r[kind + "_title"]) & (sub["Release Year"] == r[kind + "_year"])]
        if len(hit) != 1:
            print("PROBLEM:", kind, r[kind + "_title"], r[kind + "_year"], "-> matches:", len(hit))
print("checked", len(pairs), "pairs")

PROBLEM: original Pokkiri 2006 -> matches: 0
PROBLEM: remake Shankar Dada Zindabad 2007 -> matches: 0
PROBLEM: remake Khaidhi No. 150 2017 -> matches: 0
checked 5 pairs


In [9]:
import os
n_pairs = len(pd.read_csv("../data/remake_pairs.csv")) if os.path.exists("../data/remake_pairs.csv") else 0
text = f"""Project: movie plot similarity engine (Bollywood vs Hollywood/South Indian remakes).
Deadline: Nov 24, 2026. Me: beginner, 1-1.5 h/day, Windows, target = data scientist.
Teach me concepts and syntax before code; say which file each step goes in; give copy-paste steps.
Data: Kaggle Wikipedia Movie Plots; subset American, Bollywood, Tamil, Telugu, Malayalam, Kannada.
Cleaned: plots >= 50 words, duplicates removed. Rows: {len(sub)}. Saved: data/clean.csv.
Remake pairs so far: {n_pairs} in data/remake_pairs.csv.
Notebooks: 01_explore_and_clean (done), 02_embed_and_search (next).
Plan: D3 embeddings, D4 search + more pairs, D5 evaluation, D6 app, D7 deploy + README. Nov 24 = buffer.
Last blocker: none yet.
"""
with open("../CONTEXT.md", "w", encoding="utf-8") as f:
    f.write(text)
print(text)

Project: movie plot similarity engine (Bollywood vs Hollywood/South Indian remakes).
Deadline: Nov 24, 2026. Me: beginner, 1-1.5 h/day, Windows, target = data scientist.
Teach me concepts and syntax before code; say which file each step goes in; give copy-paste steps.
Data: Kaggle Wikipedia Movie Plots; subset American, Bollywood, Tamil, Telugu, Malayalam, Kannada.
Cleaned: plots >= 50 words, duplicates removed. Rows: 24076. Saved: data/clean.csv.
Remake pairs so far: 5 in data/remake_pairs.csv.
Notebooks: 01_explore_and_clean (done), 02_embed_and_search (next).
Plan: D3 embeddings, D4 search + more pairs, D5 evaluation, D6 app, D7 deploy + README. Nov 24 = buffer.
Last blocker: none yet.

